# EMHA: Kaggle bootstrap (pipeline run)


Bootstrap only: this notebook clones the repo, installs, sets paths, and calls
modules (CLAUDE.md). Pipeline logic, model code, and every hyperparameter live
in `src/` (`src/utils/config.py`); never add them here. If `pip` upgrades torch,
restart the runtime and run the cells again from the top.

Needs: Internet on, the dataset `emha-crops` attached, and the Kaggle secret `GITHUB_TOKEN` attached to this notebook.

In [ ]:
# Notebook parameters
SUBSET = "pilot"  # "pilot" or None (all participants)
ANALYSIS = "primary"  # "primary" | "full"
DEVICE = "cuda"  # "cuda" | "cpu"
REPO = "Yeeejj/EMHA"
REPO_DIR = "/tmp/EMHA"  # outside /kaggle/working, so the clone is not saved as output
DATA_ROOT = "/kaggle/input/emha-crops"
OUTPUT_ROOT = "/kaggle/working"

In [ ]:
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")

# Full clone with all history and tags (never shallow): the protocol-frozen tag
# check needs them. The token goes in a one-off HTTP header, so it is never
# written to .git/config, a URL, or the output.
import base64
import os
import subprocess

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already exists; using it as is")
else:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    subprocess.run(
        ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}",
         "clone", f"https://github.com/{REPO}.git", REPO_DIR],
        check=True,
    )
    del auth
del token
os.chdir(REPO_DIR)

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
# Record exactly what ran: commit, protocol tag, torch / CUDA.
!git rev-parse HEAD
!git describe --tags --always
!git tag --list protocol-frozen
import torch

print("torch", torch.__version__, "| CUDA build", torch.version.cuda,
      "| CUDA available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import sys

os.environ["EMHA_DATA_ROOT"] = DATA_ROOT
os.environ["EMHA_OUTPUT_ROOT"] = OUTPUT_ROOT
sys.path.insert(0, REPO_DIR)
if not os.path.isfile(os.path.join(DATA_ROOT, "metadata", "labels.csv")):
    raise SystemExit(f"{DATA_ROOT} is missing; attach the emha-crops dataset")
print(open(os.path.join(DATA_ROOT, "MANIFEST.txt")).read())
print({"SUBSET": SUBSET, "ANALYSIS": ANALYSIS, "DEVICE": DEVICE})

In [ ]:
# Guard until G5 exists: stop here with a clear message instead of a traceback.
import importlib.util

if importlib.util.find_spec("src.pipeline_runner") is None:
    raise SystemExit("src.pipeline_runner (G5) is not built yet; nothing was run")

In [ ]:
args = ["--analysis", ANALYSIS, "--device", DEVICE]
if SUBSET:
    args += ["--subset", SUBSET]
!python -m src.pipeline_runner {" ".join(args)}
if _exit_code != 0:
    raise SystemExit(f"pipeline_runner failed (exit {_exit_code})")

In [ ]:
# Zip results and models for download from the notebook's Output tab.
import shutil

for folder in ("results", "models"):
    if os.path.isdir(os.path.join(OUTPUT_ROOT, folder)):
        archive = shutil.make_archive(
            os.path.join(OUTPUT_ROOT, f"emha_{folder}"), "zip", OUTPUT_ROOT, folder
        )
        print("written", archive)